# Class 2 - The Agent Loop — Plan, Act, Observe

**Week 5: Introduction to AI Agents**

### Learning objectives
By the end of this notebook you will be able to:
- Define a LangChain tool with `@tool` and clear docstring descriptions.
- Run Plan → Act → Observe using `create_agent` + Groq (`ChatGroq`).
- Explain why a loop beats a one-shot answer when tools are involved.
- Cap the loop with `recursion_limit` so a stuck agent cannot burn quota.

**Stack note:** We use **LangChain** + **Groq** in class. **LlamaIndex** can express the same tool-use pattern with its own agent abstractions — pick one stack per project and stay consistent.

## Setup

```bash
export GROQ_API_KEY="gsk-..."
```

Install the minimal LangChain + Groq stack:

In [ ]:
!pip install -q langchain langchain-groq langgraph

In [ ]:
import os

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
try:
    from google.colab import userdata
    GROQ_API_KEY = GROQ_API_KEY or userdata.get("GROQ_API_KEY")
except Exception:
    pass

if not GROQ_API_KEY:
    print(
        "No GROQ_API_KEY found.\n"
        "Set it in your environment or add a Colab secret named GROQ_API_KEY.\n"
        "Agent cells will skip until a key is available."
    )
else:
    print("Found GROQ_API_KEY. LangChain + Groq demos are ready.")

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_groq import ChatGroq


def make_llm():
    if not GROQ_API_KEY:
        return None
    return ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0,
        api_key=GROQ_API_KEY,
    )


def build_agent(tools, system_prompt=None, max_turns=8):
    """Build a LangChain tool-calling agent. max_turns maps to recursion_limit on invoke."""
    llm = make_llm()
    if llm is None:
        return None
    prompt = system_prompt or (
        "You are a careful assistant. Use tools when they help. "
        "After you have enough observations, give a final answer."
    )
    agent = create_agent(model=llm, tools=tools, system_prompt=prompt)
    agent._class2_max_turns = max_turns  # stash for run_agent
    return agent


def run_agent(agent, question: str, thread_id: str = "class2"):
    if agent is None:
        print("Skipping — no GROQ_API_KEY / agent.")
        return None
    max_turns = getattr(agent, "_class2_max_turns", 8)
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"recursion_limit": max_turns},
    )
    messages = result.get("messages", [])
    for m in messages:
        role = getattr(m, "type", m.__class__.__name__)
        content = getattr(m, "content", str(m))
        print(f"--- {role} ---")
        print(content)
        print()
    return result

## 1. Define a tool

The model never executes Python itself. It **requests** a tool call; **your runtime** runs the function and returns an observation.

Write a clear docstring — vague descriptions cause wrong or missing tool calls.

In [ ]:
@tool
def get_weather(location: str) -> str:
    """Return a short weather summary for a city name like Kathmandu or Pokhara."""
    catalog = {
        "kathmandu": "temp_c=14, condition=rain",
        "pokhara": "temp_c=18, condition=clear",
    }
    key = location.strip().lower()
    if key not in catalog:
        return f"No weather data for '{location}'. Try Kathmandu or Pokhara."
    return catalog[key]


print(get_weather.invoke({"location": "Kathmandu"}))

## 2. Plan → Act → Observe with `create_agent`

Ask a question that needs the tool. Watch the printed message trace: the assistant plans a tool call, the tool acts, then the assistant observes and answers.

In [ ]:
weather_agent = build_agent(tools=[get_weather], max_turns=6)
run_agent(
    weather_agent,
    "What's the weather in Kathmandu, and should I bring a jacket?",
)

## 3. Why loop instead of one shot?

Without tools, the model guesses. With a loop, it can fetch an observation before advising. Re-run the same question mentally: a one-shot model might invent 22°C and sunshine; the agent above is grounded in `get_weather`.

In [ ]:
# Optional contrast: plain ChatGroq with no tools (guessing allowed)
llm = make_llm()
if llm is None:
    print("Skipping — no key.")
else:
    guess = llm.invoke("What's the weather in Kathmandu right now? One sentence.")
    print("ONE-SHOT (no tools):", guess.content)

## 4. Cap the loop

`recursion_limit` stops runaway Plan→Act→Observe cycles. Start strict in class (6–8).

In [ ]:
safe_agent = build_agent(tools=[get_weather], max_turns=4)
run_agent(safe_agent, "Weather in Pokhara — jacket or not?")

## Closing

LangChain handled tool routing; Groq powered the decisions. Class 3 adds chat memory and a multi-tool agent you can extend.

**Next:** Class 3 — Building a Simple Agent.

## Challenges

Implement each tool / safeguard with LangChain `@tool` and rebuild the agent so your new tools are included.

### Challenge 01 — `convert_currency`
Add `@tool def convert_currency(amount: float, from_currency: str, to_currency: str) -> str` using a small hard-coded rate table. Wire it into `build_agent` and ask a conversion question.

In [ ]:
# TODO: define convert_currency, rebuild agent with [get_weather, convert_currency], run_agent(...)
pass

### Challenge 02 — `get_time`
Write `@tool def get_time(timezone: str) -> str` (fake is fine: return a fixed ISO-like string per timezone). Wire it in and test.

In [ ]:
# TODO
pass

### Challenge 03 — Validate arguments
Inside a tool, reject bad input (e.g. negative `amount`) by returning an error string the agent can observe — do not raise uncaught exceptions.

In [ ]:
# TODO
pass

### Challenge 04 — `max_turns`
Rebuild with `max_turns=3` (recursion_limit) and confirm the agent still answers a simple weather question.

In [ ]:
# TODO
pass

### Challenge 05 (stretch) — Two tools in one question
Ask one question that should call both weather and currency (or time) tools before the final answer. Print the full message trace.

In [ ]:
# TODO
pass